# Model Training

Document model decisions and test training snippets

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 04/03/2026   | Martin | Created   | Notebook created. Describe metrics selected | 

# Content

* [Introduction](#introduction)

# Introduction

Board game recommendation system using the two-tower architecture with a reranker.

# Metrics Used

- __Recall@K__ - Information retrieval and recommendation metrics that measures the proportion of relevant items found in the top-K results. Results here are not rank aware i.e does not matter which position in the top-K the item was found in, as long as it's inside there
$$
\frac{Relevant\ Items\ in\ Top\ K}{Total\ Relevant\ Items}
$$

- __Normalized Discounted Cumulative Gains (NDCG@K)__ - Measures the gain of an item based on its relevantce grade and position in the top-K results. It rewards placing highly relevant items at the top of the list
  * $rel_i$: Relevance score (e.g whether the item showed up in the top-K spots)
$$
NDCG@K = \frac{DCG@K}{IDCG@K} \\
DCG@K = \sum_{k=1}^{K}\frac{rei_i}{log_2(i+1)} \\
IDCG@K = Maximum\ achievable\ DCG
$$
- __Mean Reciprocal Rank (MRR)__ - Ranking quality metrics that considers the position of the first relevant item in the ranked list. Range from 0-1 where "1" is the first relevant item being at the top of the list
  * $U$: Total number of users
$$
MRR = \frac{1}{U}\sum_{u=1}{U}\frac{1}{rank_i}
$$
- __HitRate@K__ - Measures the number of users whom at least one relevant item is present in the top-K list. Coarse metric
- __Mean Average Precision (MAP@K)__ - Ranking quality metric that considers the number of relevant recommendations and their position in the list.
  * $N$: Total number of relevant items in the top-K results
  * Divide the total number of relevant items by the total number of items until this position
$$
MAP@K = \frac{1}{U}\sum_{u=1}^U AP@K_u \\
AP@K = \frac{1}{N}\sum_{k=1}^K Precision(k) \times rel(k)
$$

- __Coverage@K__ - Measures the percentage of unique items that was recommended over the total number of items. Evalutes the diversity of items
$$
Coverage@K = \frac{Number\ of\ unique\ items\ recommended\ in\ top-K}{Total\ number\ of\ items\ in\ catalog}
$$

In [1]:
import duckdb

# Data Loading

## Converting to DuckDB calls

Original code uses CSVs, converting to DuckDB calls

In [2]:
DUCKDB_PATH = "../data/bgg_db.duckdb"

In [3]:
con = duckdb.connect(DUCKDB_PATH)

In [4]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                                                                                                                             column_names                                                                                                                                                       

In [5]:
games = con.sql("SELECT * FROM bgg.games").df()
comments = con.sql("SELECT * FROM bgg.comments").df()

In [9]:
games.columns

Index(['id', 'name', 'description', 'year_published', 'min_players',
       'max_players', 'suggested_players', 'playing_time', 'min_playing_time',
       'max_playing_time', 'min_age', 'suggested_age', 'category_ids',
       'categories', 'mechanic_ids', 'mechanics', 'family_ids', 'families',
       'designer_ids', 'designers', 'artist_ids', 'artists', 'num_ratings',
       'rating', 'bayes_rating', 'complexity'],
      dtype='str')

In [ ]:
%load_ext watermark
%watermark